In [1]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import imgaug as ia
import imgaug.augmenters as iaa
import pickle
import json
from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.svm import SVC

ia.seed(1)

Number of image each label

In [2]:
path = "samples"
labels = os.listdir(path)

print('labels:\n',labels)

features = []
for l in labels:
    f = os.listdir(os.path.join(path,l))
    features.append(len(f))

# find unique value
n = 1
for f in list(set(features)):
    n *=f
    
print('features:\n',features)
print('n= ',n)

ms = []
for f in features:
    ms.append(int(n/f))
print('ms:\n',ms)

labels:
 ['3', '18', '6', '23', '19', '24', '20', '17', '16', '11', '21', '1', '7', '15', '25', '8', '4', '5', '13', '14', '10', '22', '9', '26', '12', '2']
features:
 [2, 1, 3, 2, 3, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 1, 1, 1, 2, 2, 6, 1, 2]
n=  36
ms:
 [18, 36, 12, 18, 12, 18, 36, 18, 36, 18, 18, 18, 18, 18, 18, 36, 18, 18, 36, 36, 36, 18, 18, 6, 36, 18]


In [3]:
N = 1
n*N

36

Data Augmentation

In [4]:
seq = iaa.Sequential([
    iaa.Fliplr(0.5), # horizontal flips
    iaa.Crop(percent=(0, 0.1)), # random crops
    # Small gaussian blur with random sigma between 0 and 0.5.
    # But we only blur about 50% of all images.
    iaa.Sometimes(
        0.5,
        iaa.GaussianBlur(sigma=(0, 0.5))
    ),
    # Strengthen or weaken the contrast in each image.
    iaa.LinearContrast((0.75, 1.5)),
    # Add gaussian noise.
    # For 50% of all images, we sample the noise once per pixel.
    # For the other 50% of all images, we sample the noise per pixel AND
    # channel. This can change the color (not only brightness) of the
    # pixels.
    iaa.AdditiveGaussianNoise(loc=0, scale=(0.0, 0.05*255), per_channel=0.5),
    # Make some images brighter and some darker.
    # In 20% of all cases, we sample the multiplier once per channel,
    # which can end up changing the color of the images.
    iaa.Multiply((0.8, 1.2), per_channel=0.2),
    # Apply affine transformations to each image.
    # Scale/zoom them, translate/move them, rotate them and shear them.
    iaa.Affine(
        scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
        translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
        rotate=(-10, 10),
        shear=(-4, 4)
    )
], random_order=True) # apply augmenters in random order

In [5]:
DIM=227
N = 20 #100 overlimit

path ="samples"
sub = 'train'

print('creating training dataset')

if not os.path.isdir(f'{sub}'):
        os.mkdir(f'{sub}')
        
for i,label in enumerate(labels):
    
    # create train/label
    if not os.path.isdir(f'{sub}/{label}'):
        os.mkdir(f'{sub}/{label}')
    
    label_dir = os.path.join(path,label)
    imgs = os.listdir(label_dir)
    for im in imgs:
        img = cv2.imread(os.path.join(label_dir,im))
        # resize image
        img = cv2.resize(img,(DIM,DIM), interpolation = cv2.INTER_AREA)
        # duplicate
        images = np.array([ img for _ in range(N*ms[i])],dtype=np.uint8)
        # augmentation
        images_aug = seq(images=images)
        
        for j in range(N*ms[i]):
            cv2.imwrite(f"{sub}/{label}/{im.strip('.jpg')}_{j}.jpg",images_aug[j])
            

creating training dataset


In [ ]:
N = 20
path ="samples"
sub = 'test'

print('creating testing dataset')

if not os.path.isdir(f'{sub}'):
        os.mkdir(f'{sub}')
        
for i,label in enumerate(labels):
    
    # create train/label
    if not os.path.isdir(f'{sub}/{label}'):
        os.mkdir(f'{sub}/{label}')
    
    label_dir = os.path.join(path,label)
    imgs = os.listdir(label_dir)
    for im in imgs:
        img = cv2.imread(os.path.join(label_dir,im))
        # resize image
        img = cv2.resize(img,(DIM,DIM), interpolation = cv2.INTER_AREA)
        # duplicate
        images = np.array([ img for _ in range(N*ms[i])],dtype=np.uint8)
        # augmentation
        images_aug = seq(images=images)
        
        for j in range(N*ms[i]):
            cv2.imwrite(f"{sub}/{label}/{im.strip('.jpg')}_{j}.jpg",images_aug[j])

creating testing dataset


Load dictionary of label

In [ ]:
with open('dicts.json','r') as f:
    dicts = json.load(f)

In [ ]:
dic ={}
for d in dicts:
    dic[int(d)] = dicts[d]['device']
dic

Train

In [ ]:
print('loading training dataset') 
X = []
y = [] #number

path="train"
IMG_SIZE=100

labels = os.listdir(path)

for l in labels:
    imgs = os.listdir(os.path.join(path,l))
    for im in imgs:
        img = cv2.imread(os.path.join(path,l,im))
        img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
        X.append(img)
        y.append(int(l))
        
# shuffle
X,y = shuffle(X,y)

X = np.array(X).reshape(len(X),-1)
# norm
X = X/255.0
y = np.array(y)

X = np.array(X).reshape(len(X),-1)
# norm
X = X/255.0
y = np.array(y)

# split
X_train, X_val, y_train, y_val = train_test_split(X,y)
print("X_train: ",X_train.shape)
print("X_val: ",X_val.shape)
print("y_train: ",y_train.shape)
print("y_val: ",y_val.shape)

In [ ]:
svc = SVC(kernel='linear',gamma='auto') #linear
svc.fit(X_train, y_train)

print('testing...')

y2 = svc.predict(X_val)

# calc accuracy
print("Accuracy on validation dataset is",accuracy_score(y_val,y2))

print("Accuracy on validation dataset is")
print(classification_report(y_val,y2))

# save the model to disk
filename = 'ppocr/obj_cla.sav'
pickle.dump(svc, open(filename, 'wb'))

Testing

In [ ]:
# load model
loaded_model = pickle.load(open("ppocr/obj_cla.sav", 'rb'))

In [ ]:
# random test
IMG_SIZE=100
path = 'samples'
labels = os.listdir(path)
label = random.choice(labels)
imgs = os.listdir(os.path.join(path,label))
im = random.choice(imgs)
img = cv2.imread(os.path.join(path,label,im))
img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
plt.imshow(img)
img = img/255.0
img = img.reshape(1,-1)
y= loaded_model.predict(img)
plt.title("pred: " + dic[int(i.strip('.jpg'))] +"\n(" + dic[int(y)] + ")")